In [ ]:
!pip install tabpfn --no-index --find-links=file:///kaggle/input/pip-packages-icr/pip-packages

In [ ]:
import sys
sys.path.append('/kaggle/input/icr-utils')

import os, pickle, gc, warnings, copy
from glob import glob
# warnings.filterwarnings('ignore')

from tqdm import tqdm
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder

from utils import save_pickle, load_pickle

In [ ]:
ORI_DIR = '/kaggle/input/icr-identify-age-related-conditions'
CONFIG = {
    'integerized': True
    , 'encoded': True
    , 'sscaled': False
    , 'imputing_strategy': 'median' # mean, median, constant
    , 'epsilon_cols': ['Epsilon_ordinal'] # Epsilon cols to include
    , 'x_mul_y': True
    , 'x_div_y': True
    , 'drop_cols': ['EH', 'DV', 'BZ']
}

save_pickle(CONFIG, 'config.pkl')

In [ ]:
df_train = pd.read_csv(os.path.join(ORI_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(ORI_DIR, 'test.csv'))

# Correlation

In [ ]:
f = [c for c in df_train.columns if c not in ['Id', 'Class', 'EJ']]
corr = df_train[f].corr()
for r in range(len(f)):
    for c in range(0, r+1):
        corr.iloc[r, c] = None
corr.to_csv('correlation.csv', index=False)

In [ ]:
high_corr = {}
threshold = 0.5
for idx, c0 in enumerate(f):
    for c1 in f[idx+1:]:
        val = corr.loc[c0, c1]
        if val >= threshold:
            high_corr[f'{c0}_{c1}'] = val
high_corr = pd.Series(high_corr).sort_values(ascending=False)
high_corr.to_csv('high_correlation.csv', index=False)
high_corr

# `NaN`

In [ ]:
nulls = df_train.isnull().sum()
nulls.loc[nulls > 0]

# Feature Engineering

In [ ]:
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
greeks['Epsilon_dt'] = greeks.Epsilon.replace('Unknown', np.nan)
greeks['Epsilon_dt'] = pd.to_datetime(greeks.Epsilon_dt, format='%m/%d/%Y')

min_date = greeks['Epsilon_dt'].min()
greeks['Epsilon_year'] = greeks.Epsilon_dt.dt.year
greeks['Epsilon_month'] = greeks.Epsilon_dt.dt.month
greeks['Epsilon_weekday'] = greeks.Epsilon_dt.dt.weekday
greeks['Epsilon_days_from_min'] = (greeks['Epsilon_dt'] - min_date).dt.days
greeks['Epsilon_ordinal'] = greeks.Epsilon.replace('Unknown', np.nan)
valid_date = (greeks.Epsilon != 'Unknown')
greeks.loc[valid_date, 'Epsilon_ordinal'] = greeks.Epsilon_dt.loc[valid_date].map(lambda x: x.toordinal())

greeks.head()

In [ ]:
%%writefile feature_engineering.py
from itertools import combinations
from datetime import datetime as dt
import copy

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,StandardScaler

# Integerized ---> https://www.kaggle.com/code/raddar/convert-icr-data-to-integers
int_denominators = {
 'AB': 0.004273,
 'AF': 0.00242,
 'AH': 0.008709,
 'AM': 0.003097,
 'AR': 0.005244,
 'AX': 0.008859,
 'AY': 0.000609,
 'AZ': 0.006302,
 'BC': 0.007028,
 'BD': 0.00799,
 'BN': 0.3531,
 'BP': 0.004239,
 'BQ': 0.002605,
 'BR': 0.006049,
 'BZ': 0.004267,
 'CB': 0.009191,
 'CC': 6.12e-06,
 'CD': 0.007928,
 'CF': 0.003041,
 'CH': 0.000398,
 'CL': 0.006365,
 'CR': 7.5e-05,
 'CS': 0.003487,
 'CU': 0.005517,
 'CW': 9.2e-05,
 'DA': 0.00388,
 'DE': 0.004435,
 'DF': 0.000351,
 'DH': 0.002733,
 'DI': 0.003765,
 'DL': 0.00212,
 'DN': 0.003412,
 'DU': 0.0013794,
 'DV': 0.00259,
 'DY': 0.004492,
 'EB': 0.007068,
 'EE': 0.004031,
 'EG': 0.006025,
 'EH': 0.006084,
 'EL': 0.000429,
 'EP': 0.009269,
 'EU': 0.005064,
 'FC': 0.005712,
 'FD': 0.005937,
 'FE': 0.007486,
 'FI': 0.005513,
 'FR': 0.00058,
 'FS': 0.006773,
 'GB': 0.009302,
 'GE': 0.004417,
 'GF': 0.004374,
 'GH': 0.003721,
 'GI': 0.002572
}

# greeks
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
valid_date = (greeks.Epsilon != 'Unknown')
greeks['Epsilon_dt'] = greeks.Epsilon.replace('Unknown', np.nan)
greeks['Epsilon_ordinal'] = greeks.Epsilon_dt.copy()
greeks['Epsilon_dt'] = pd.to_datetime(greeks.Epsilon_dt, format='%m/%d/%Y')
greeks.loc[valid_date, 'Epsilon_ordinal'] = greeks.Epsilon_dt.loc[valid_date].map(lambda x: x.toordinal())
greeks['Epsilon_ordinal'] = greeks.Epsilon_ordinal.astype(float)

greeks['Epsilon_year'] = greeks.Epsilon_dt.dt.year
greeks['Epsilon_month'] = greeks.Epsilon_dt.dt.month
greeks['Epsilon_weekday'] = greeks.Epsilon_dt.dt.weekday

EPS_COL_DEFAULT = [
    'Epsilon_year', 'Epsilon_month', 'Epsilon_weekday'
    , 'Epsilon_days_from_min', 'Epsilon_ordinal'
]

class Imputer():
    def __init__(self, features, strategy='mean'):
        self.strategy = strategy
        self.strategy_constant = (
            isinstance(self.strategy, int)
            or isinstance(self.strategy, float)
        )
        self.features = features
    
    def fit(self, df):
        if self.strategy == 'mean':
            self.default_value = df[self.features].mean()
        elif self.strategy == 'median':
            self.default_value = df[self.features].median()
        elif self.strategy_constant:
            self.default_value = self.strategy
        return df
    
    def transform(self, df):
        if (
            (self.strategy in ('mean', 'median'))
            or self.strategy_constant
        ):
            df.fillna(self.default_value, inplace=True)
        elif self.strategy is None:
            pass
        return df
    
    def fit_transform(self, df):
        df = self.fit(df)
        df = self.transform(df)
        return df

class FeatureEncoder():
    def __init__(self, encoded_cols):
        self.encoders = {c: LabelEncoder() for c in encoded_cols}
        self.encoded_cols = encoded_cols
        
    def fit(self, df):
        for col in self.encoded_cols:
            self.encoders[col].fit(df[col])
        
    def transform(self, df):
        for col in self.encoded_cols:
            df[col] = self.encoders[col].transform(df[col])
        return df
    
class FeatureScaler():
    def __init__(self, scaled_cols):
        self.scaler = StandardScaler()
        self.scaled_cols = scaled_cols
        
    def fit(self, df):
        self.scaler.fit(df[self.scaled_cols])
        
    def transform(self, df):
        df[self.scaled_cols] = self.scaler.transform(df[self.scaled_cols])
        return df
    
    def fit_transform(self, df):
        self.fit(df)
        df = self.transform(df)
        return df

class FeatureEngineer():
    def __init__(
        self
        , integerized=True
        , imputing_strategy='mean' # mean
        , x_mul_y=True
        , x_div_y=True
        , encoded=True
        , sscaled=True
        , epsilon_cols=[]
        , drop_cols=[]
    ):
        self.integerized = integerized
        self.imputing_strategy = imputing_strategy
        self.imputing = self.imputing_strategy is not None
        self.int_denominators = int_denominators
        self.encoded = encoded
        self.sscaled = sscaled
        self.greeks = greeks
        self.is_fitted_ = False
        self.x_mul_y = x_mul_y
        self.x_div_y = x_div_y
        self.drop_cols = drop_cols
        
        self.cat_cols = ['EJ'] if 'EJ' not in self.drop_cols else []
        self.id_class_cols = ['Id', 'Class']
        self.num_cols = [
            c for c in ['FL', 'GL'] + list(self.int_denominators.keys())
            if c not in self.drop_cols
        ]
        # Epsilon
        self.epsilon_cols_default = EPS_COL_DEFAULT
        self.epsilon_cols = np.intersect1d(epsilon_cols, self.epsilon_cols_default)
        self.epsilon_cols = self.epsilon_cols.tolist()
        self.epsilon_cols_drop = [
            c for c in self.epsilon_cols_default if c not in self.epsilon_cols
        ]
        
    def _strip_colnames(self, df):
        df = df.copy()
        df.columns = df.columns.str.strip()
        return df
    
    
    def _preprocess_epsilon(self, df, is_training=True):
        # Eplison time
        if is_training:
            if 'Epsilon_year' not in df.columns:
                cols = [
                    'Id', 'Epsilon_dt', 'Epsilon_year'
                    , 'Epsilon_month', 'Epsilon_weekday', 'Epsilon_ordinal'
                ]
                df = df.merge(self.greeks.loc[:, cols], on='Id', how='left')
            if not self.is_fitted_:
                # Calcualte default Epsilon
                self.min_Epsilon = df['Epsilon_dt'].min()
                self.Epsilon_year_dflt = int(df['Epsilon_year'].max()) + 1
                self.Epsilon_month_dflt = int(df['Epsilon_month'].median())
                self.Epsilon_weekday_dflt = df['Epsilon_weekday'].median()
                self.Epsilon_ordinal_dflt = df['Epsilon_ordinal'].max() + 1
                
                ym = f'{self.Epsilon_year_dflt}/{self.Epsilon_month_dflt}'
                for d in range(1,31):
                    date_string = f'{ym}/{d}'
                    date = dt.strptime(date_string, '%Y/%m/%d')
                    if date.weekday() == self.Epsilon_weekday_dflt:
                        break
                self.Epsilon_days_dflt = (date - self.min_Epsilon).days
            
            df['Epsilon_days_from_min'] = (df['Epsilon_dt'] - self.min_Epsilon).dt.days
            df = df.drop(columns='Epsilon_dt')
        else:
            df['Epsilon_year'] = self.Epsilon_year_dflt
            df['Epsilon_month'] = self.Epsilon_month_dflt
            df['Epsilon_weekday'] = self.Epsilon_weekday_dflt
            df['Epsilon_days_from_min'] = self.Epsilon_days_dflt
            df['Epsilon_ordinal'] = self.Epsilon_ordinal_dflt
        df = df.drop(columns=self.epsilon_cols_drop)
        return df
        
    def _preprocess(self, df, is_training=True):
        df = self._strip_colnames(df)
        df = self._preprocess_epsilon(df, is_training=is_training)
        return df
    
    def _add_interaction_fts(self, df):
        new_cols = []
        for X, Y in combinations(self.combination_cols, 2):
            if self.x_mul_y:
                tmp = pd.Series(df[X] * df[Y], name=f'{X}_mul_{Y}')
                new_cols.append(tmp / tmp.max())
            if self.x_div_y:
                new_cols.append(pd.Series(df[X] / df[Y].replace({0:np.nan}), name=f'{X}_div_{Y}'))
                new_cols.append(pd.Series(df[Y] / df[X].replace({0:np.nan}), name=f'{Y}_div_{X}'))
        df = pd.concat([df] + new_cols, axis=1)
        return df
    
    def _fit(self, df):
        df = df.copy()
        # Check
        if self.is_fitted_:
            raise Exception('This FeatureEngineer object is already fitted!')
            
        self.features = self.num_cols + self.epsilon_cols
        self.imputed_cols = self.num_cols + self.epsilon_cols
        self.sscaled_cols = self.num_cols.copy()
        self.combination_cols = self.num_cols.copy()
        
        if self.encoded:
            self.encoder = FeatureEncoder(encoded_cols=self.cat_cols)
            self.encoder.fit(df)
            self.imputed_cols += self.cat_cols
            self.features += self.cat_cols
        
        if self.imputing:
            self.imputer = Imputer(self.imputed_cols, strategy=self.imputing_strategy)
            self.imputer.fit(df)
        
        self.X_combined_Y_features = []
        if self.x_mul_y:
            self.X_combined_Y_features += [
                f'{X}_mul_{Y}' for X, Y in combinations(self.combination_cols, 2)
            ]
        if self.x_div_y:
            div_fts = [
                name for X, Y in combinations(self.combination_cols, 2)
                for name in [f'{X}_div_{Y}', f'{Y}_div_{X}']
            ]
            self.X_combined_Y_features += div_fts
        if self.imputing and (self.x_mul_y or self.x_div_y):
            df_extra_cols = self._add_interaction_fts(df)
            self.div_imputer = Imputer(
                self.X_combined_Y_features
                , strategy=self.imputing_strategy
            )
            self.div_imputer.fit(df_extra_cols)
            
        self.sscaled_cols += self.X_combined_Y_features
        self.features += self.X_combined_Y_features
        
        self.is_fitted_ = True
        return df
        
    def _transform(self, df):
        # Check if fitted
        if not self.is_fitted_:
            raise Except('FeatureEngineer module not fitted!')
        df = df.copy()
        if self.integerized:
            for k, v in self.int_denominators.items():
                df[k] = np.round(df[k]/v,1)
                
        if self.encoded:
            df = self.encoder.transform(df)
            
        if self.imputing:
            df = self.imputer.transform(df)
        
        df = self._add_interaction_fts(df)
        if self.imputing and (self.x_mul_y or self.x_div_y):
            df = self.div_imputer.transform(df)
        
        return df
    
    def _postprocess(self, df, is_fitting=True):
        if self.sscaled:
            if is_fitting:
                self.scaler = FeatureScaler(scaled_cols=self.sscaled_cols)
                self.scaler.fit(df)
            df = self.scaler.transform(df)
        
        return df
    
    def fit(self, df, is_training=True):
        df = self._preprocess(df, is_training=is_training)
        df = self._fit(df)
        return df
    
    def transform(self, df, is_training=True):
        df = self._preprocess(df, is_training=is_training)
        df = self._transform(df)
        df = self._postprocess(df, is_fitting=False)
        return df
    
    def fit_transform(self, df, is_training=True):
        df = self._preprocess(df, is_training=is_training)
        df = self._fit(df)
        df = self._transform(df)
        df = self._postprocess(df, is_fitting=True)

        return df
    
def check_original_df(df, FE):
    df = df.copy()
    df = FE._strip_colnames(df)
    print('Null count:', df[FE.num_cols + FE.cat_cols].isna().sum().sum())
    print(f'N columns: {len(df.columns)}')
    print('EJ in features:', 'EJ' in df.columns)  
    
def check_transformed_df(df, FE):
    print('Null count:', df[FE.features].isna().sum().sum())
    print(f'N columns: {len(df.columns)}')
    print('EJ in features:', 'EJ' in FE.features)
    print('Epsilon cols: ', [c for c in df.columns if 'Epsilon' in c])

In [ ]:
from feature_engineering import FeatureEngineer, check_original_df, check_transformed_df

with open('config.pkl', 'rb') as handle:
    config = pickle.load(handle)

FE = FeatureEngineer(
    **CONFIG
)
df_train_fe = FE.fit_transform(df_train, is_training=True)
df_test_fe = FE.transform(df_test, is_training=False)

file_name = 'FeatureEngineer.pkl'
with open(file_name, 'wb') as file:
    pickle.dump(FE, file)
    print(f'Object successfully saved to "{file_name}"')

In [ ]:
check_original_df(df_train, FE)
df_train.head()

In [ ]:
check_original_df(df_test, FE)
df_test.head()

In [ ]:
check_transformed_df(df_train_fe, FE)
df_train_fe.head()

In [ ]:
df_train_fe['Epsilon_ordinal']

In [ ]:
check_transformed_df(df_test_fe, FE)
df_test_fe.head()

In [ ]:
df_test_fe['Epsilon_ordinal']

In [ ]:
print(len(FE.features))
print(len(set(FE.features)))